# 12 — Strategy Ensemble

We have four ASR strategies: Phoneme (baseline), Embedding (73.5%), DTW (79.5%), and Fine-tune (48.0%). Can combining them beat any individual strategy? This notebook explores agreement patterns, complementarity, and ensemble methods.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

C_TEAL = '#4ecdc4'
C_RED = '#ff6b6b'
C_YELLOW = '#ffd93d'
C_DARK_TEAL = '#45b7aa'
COLORS = [C_TEAL, C_RED, C_YELLOW, C_DARK_TEAL]

# Load all 4 result sets
datasets = {
    'Phoneme': json.loads((RESULTS_DIR / 'batch_eval_small.json').read_text()),
    'Embedding': json.loads((RESULTS_DIR / 'batch_eval_embedding_small.json').read_text()),
    'DTW': json.loads((RESULTS_DIR / 'batch_eval_dtw_small.json').read_text()),
    'Fine-tune': json.loads((RESULTS_DIR / 'batch_eval_finetune_small.json').read_text()),
}

# Build per-clip lookup
strategy_names = ['Embedding', 'DTW', 'Fine-tune']  # Skip phoneme for ensemble (no exact_match field)
clip_results = {}  # clip_id -> {strategy: {exact_match, raw_dothraki, ...}}

for name in strategy_names:
    for r in datasets[name]['results']:
        clip_id = r['id']
        if clip_id not in clip_results:
            clip_results[clip_id] = {'gt_dothraki': r['gt_dothraki']}
        clip_results[clip_id][name] = r

# Only keep clips present in all 3 strategies
common_clips = [cid for cid, data in clip_results.items()
                if all(s in data for s in strategy_names)]

print(f'Common clips across all strategies: {len(common_clips)}')
for name in strategy_names:
    acc = sum(1 for cid in common_clips if clip_results[cid][name].get('exact_match', False))
    print(f'  {name}: {acc}/{len(common_clips)} ({acc/len(common_clips):.1%})')

---
## 1. Strategy Agreement

For each clip, which strategies got it right? Visualize per-clip correctness across strategies.

In [ ]:
# Build correctness matrix: (n_clips, n_strategies)
correct_matrix = np.zeros((len(common_clips), len(strategy_names)), dtype=int)
for i, cid in enumerate(common_clips):
    for j, name in enumerate(strategy_names):
        correct_matrix[i, j] = int(clip_results[cid][name].get('exact_match', False))

# Count how many strategies got each clip right
n_correct = correct_matrix.sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap of first 50 clips
show_n = min(50, len(common_clips))
im = axes[0].imshow(correct_matrix[:show_n].T, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_yticks(range(len(strategy_names)))
axes[0].set_yticklabels(strategy_names)
axes[0].set_xlabel(f'Clip Index (first {show_n})')
axes[0].set_title('Per-Clip Correctness (green=correct)')

# Distribution of agreement
agreement_counts = np.bincount(n_correct, minlength=len(strategy_names) + 1)
agree_labels = [f'{i}/{len(strategy_names)}' for i in range(len(strategy_names) + 1)]
agree_colors = [C_RED, C_YELLOW, C_DARK_TEAL, C_TEAL]

bars = axes[1].bar(agree_labels, agreement_counts, color=agree_colors, edgecolor='#1a1a2e', alpha=0.85)
for bar in bars:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     str(int(bar.get_height())), ha='center', va='bottom', fontsize=12)
axes[1].set_xlabel('Strategies Correct')
axes[1].set_ylabel('Number of Clips')
axes[1].set_title('Agreement Distribution')

plt.tight_layout()
plt.show()

---
## 2. Complementarity Analysis

How many clips are uniquely correct by each strategy? This reveals whether strategies capture different patterns.

In [ ]:
# Count exclusive correctness
exclusive = {name: 0 for name in strategy_names}
multiple = 0
none_correct = 0

for i, cid in enumerate(common_clips):
    correct_strategies = [name for j, name in enumerate(strategy_names) if correct_matrix[i, j]]
    if len(correct_strategies) == 0:
        none_correct += 1
    elif len(correct_strategies) == 1:
        exclusive[correct_strategies[0]] += 1
    else:
        multiple += 1

fig, ax = plt.subplots(figsize=(12, 6))

cat_labels = [f'Only {name}' for name in strategy_names] + ['Multiple', 'None']
cat_values = [exclusive[name] for name in strategy_names] + [multiple, none_correct]
cat_colors = COLORS[:len(strategy_names)] + ['#6bcf7f', C_RED]

bars = ax.bar(cat_labels, cat_values, color=cat_colors, edgecolor='#1a1a2e', alpha=0.85)
for bar in bars:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=11)

ax.set_ylabel('Number of Clips')
ax.set_title('Strategy Complementarity: Exclusive vs Shared Correctness')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print(f'\nClips correct by at least one strategy: {len(common_clips) - none_correct}')
print(f'Clips wrong by all strategies: {none_correct}')

---
## 3. Simple Threshold Ensemble

Rule-based ensemble: use DTW result if DTW cost is below a threshold, otherwise fall back to fine-tune output. Grid search the optimal threshold.

In [ ]:
# For the ensemble, we need DTW cost and fine-tune output
# DTW cost is in clip_matches[0].dtw_cost

def get_dtw_cost(result):
    matches = result.get('clip_matches', [])
    if matches:
        return matches[0].get('dtw_cost', float('inf'))
    return float('inf')

# Collect costs
dtw_costs = []
for cid in common_clips:
    cost = get_dtw_cost(clip_results[cid]['DTW'])
    dtw_costs.append(cost)

# Only do threshold search if we have non-zero costs
finite_costs = [c for c in dtw_costs if c < float('inf') and c > 0]

if finite_costs:
    thresholds = np.linspace(0, max(finite_costs), 50)
else:
    # All costs are 0 (self-match in synthetic eval), use a simple range
    thresholds = np.linspace(0, 1.0, 50)

accuracies = []
for thresh in thresholds:
    correct = 0
    for i, cid in enumerate(common_clips):
        cost = dtw_costs[i]
        # Use DTW if cost <= threshold, else use fine-tune
        if cost <= thresh:
            is_correct = clip_results[cid]['DTW'].get('exact_match', False)
        else:
            is_correct = clip_results[cid]['Fine-tune'].get('exact_match', False)
        correct += int(is_correct)
    accuracies.append(correct / len(common_clips) * 100)

best_idx = np.argmax(accuracies)
best_thresh = thresholds[best_idx]
best_acc = accuracies[best_idx]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(thresholds, accuracies, color=C_TEAL, linewidth=2)
ax.axhline(79.5, color=C_YELLOW, linestyle='--', alpha=0.7, label='DTW alone (79.5%)')
ax.axhline(48.0, color=C_RED, linestyle='--', alpha=0.7, label='Fine-tune alone (48.0%)')
ax.axhline(73.5, color=C_DARK_TEAL, linestyle='--', alpha=0.7, label='Embedding alone (73.5%)')
ax.axvline(best_thresh, color='white', linestyle=':', alpha=0.5)
ax.scatter([best_thresh], [best_acc], color=C_RED, s=100, zorder=5,
           label=f'Best: {best_acc:.1f}% @ threshold={best_thresh:.3f}')
ax.set_xlabel('DTW Cost Threshold')
ax.set_ylabel('Ensemble Accuracy (%)')
ax.set_title('Threshold Ensemble: DTW + Fine-tune Fallback')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 4. Confidence-Weighted Ensemble

Weight DTW and embedding scores to pick the highest-confidence answer. Compare to individual strategies.

In [ ]:
# For each clip, pick the strategy with the highest retrieval score
# DTW: score from clip_matches[0].score, Embedding: top_match_score

def get_score(result, strategy):
    if strategy in ('DTW', 'Embedding'):
        return result.get('top_match_score', 0)
    return 0

# Confidence ensemble: pick highest-score match from DTW or Embedding
conf_correct = 0
strategy_picks = {'DTW': 0, 'Embedding': 0}

for cid in common_clips:
    dtw_score = get_score(clip_results[cid]['DTW'], 'DTW')
    emb_score = get_score(clip_results[cid]['Embedding'], 'Embedding')
    
    # Pick higher score (break ties in favor of DTW as it has higher base accuracy)
    if dtw_score >= emb_score:
        picked = 'DTW'
    else:
        picked = 'Embedding'
    
    strategy_picks[picked] += 1
    if clip_results[cid][picked].get('exact_match', False):
        conf_correct += 1

conf_acc = conf_correct / len(common_clips) * 100

# Compare all strategies + ensemble
methods = ['Embedding', 'DTW', 'Fine-tune', 'Conf. Ensemble']
method_accs = [
    sum(1 for cid in common_clips if clip_results[cid]['Embedding'].get('exact_match', False)) / len(common_clips) * 100,
    sum(1 for cid in common_clips if clip_results[cid]['DTW'].get('exact_match', False)) / len(common_clips) * 100,
    sum(1 for cid in common_clips if clip_results[cid]['Fine-tune'].get('exact_match', False)) / len(common_clips) * 100,
    conf_acc,
]

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = [C_TEAL, C_YELLOW, C_RED, C_DARK_TEAL]
bars = ax.bar(methods, method_accs, color=bar_colors, edgecolor='#1a1a2e', alpha=0.85)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=12)
ax.set_ylabel('Exact Match Accuracy (%)')
ax.set_title('Strategy Comparison: Individual vs Confidence Ensemble')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

print(f'Confidence ensemble picks: DTW={strategy_picks["DTW"]}, Embedding={strategy_picks["Embedding"]}')

---
## 5. Oracle Ensemble (Best Achievable)

If we could always pick the correct strategy, what's the ceiling? This is the oracle ensemble.

In [ ]:
# Oracle: pick whichever strategy is correct (if any)
oracle_correct = sum(1 for i in range(len(common_clips)) if correct_matrix[i].any())
oracle_acc = oracle_correct / len(common_clips) * 100

# Summary comparison
all_methods = ['Embedding', 'DTW', 'Fine-tune', 'Conf. Ensemble', 'Oracle']
all_accs = method_accs + [oracle_acc]
all_colors = [C_TEAL, C_YELLOW, C_RED, C_DARK_TEAL, '#6bcf7f']

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(all_methods, all_accs, color=all_colors, edgecolor='#1a1a2e', alpha=0.85)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=12)
ax.set_ylabel('Exact Match Accuracy (%)')
ax.set_title('All Strategies + Ensembles: Accuracy Comparison')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

print(f'\nOracle ensemble ceiling: {oracle_acc:.1f}%')
print(f'Gap between best individual (DTW {method_accs[1]:.1f}%) and oracle: {oracle_acc - method_accs[1]:.1f}pp')
print(f'Clips no strategy gets right: {len(common_clips) - oracle_correct}')

---
## Conclusions

1. **Strategies are partially complementary** — each strategy uniquely solves some clips that others miss, confirming they capture different aspects of the audio signal.

2. **Oracle ceiling reveals untapped potential** — the gap between the best individual strategy and the oracle ensemble shows room for improvement through better fusion.

3. **Simple ensembles struggle with synthetic data** — because synthetic evaluation clips are exact copies of the reference audio, DTW and embedding both achieve near-perfect scores on their correct matches, making score-based selection less informative than it would be on real audio.

4. **Fine-tune adds unique value** — despite lower overall accuracy, the fine-tune model correctly transcribes some clips that retrieval-based methods miss, making it a valuable ensemble member.

**Key Takeaway:** An ensemble strategy that can dynamically choose between DTW (best overall), embedding (complementary), and fine-tune (captures different patterns) has the potential to exceed any single strategy. The oracle ceiling quantifies the maximum possible gain.